# Lozano-Smith Algorithm

In [1]:
import pyomo.environ as pyo
from pyomo.opt import SolverFactory
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
#from bi_level import * 

EXEC_PATH = '/Applications/CPLEX_Studio221/cplex/bin/x86-64_osx/cplex'

In [2]:
data = pd.read_excel('../Instances/input.xlsx', None)

## Initialization

In [3]:
def INIT(data):
    
    model = pyo.ConcreteModel() # create model
    mult = 100 # <--------------- multiplication factor for making the variable integer

    # data
    t = len(data['Prices']['Spot Market'])
    PI = data['Prices']['Spot Market'].values.flatten() 
    PI_cap = data['Prices']['Capacity price'].values.flatten() 
    l = len(data['Power price']['Power'])

    X_up = data['Average prices']['Max price'].values.flatten() 
    X_low = data['Average prices']['Min price'].values.flatten() 
    X_avg = 0.12
    Mi_up = data['Average prices']['Max cap'].values.flatten() 
    Mi_low = data['Average prices']['Min cap'].values.flatten() 
    Mi_avg = 0.01
    
    p = len(data['Periods']['Period'])
    Q_begin = data['Periods']['Begin'].tolist()
    Q_end = data['Periods']['End'].tolist()
    Q_len = data['Periods']['Len'].tolist()

    i = len(data['Trip time']['Time begin (min)'])
    k = len(data['Buses']['Bus (kWh)'])
    n = len(data['Chargers']['Charger (kWh/min)'])
    T_start = data['Trip time']['Time begin (min)'].tolist()
    T_start = [int(x) for x in T_start]
    T_end = data['Trip time']['Time finish (min)'].tolist()
    T_end = [int(x) for x in T_end]
    alpha = data['Chargers']['Charger (kWh/min)'].tolist()
    beta = data['Chargers']['Charger (kWh/min)'].tolist()
    gama = data['Energy consumption']['Uncertain energy (kWh/km*min)'].tolist()

    ch_eff = 0.90
    E_0 = 0.2
    E_min = 0.2
    E_max = 1
    E_end = 0.2
    C_bat = data['Buses']['Bus (kWh)'].tolist()
    T_final = t

    ### ROBUST ENERGY ###

    gama_deviation = data['Energy consumption']['Maximum deviation (kWh/km*min)'].tolist()
    gama_capital = 0 # 0 < Gama < number of routes (worst case)

    #####################

    # sets
    model.P = pyo.RangeSet(p) # set of periods
    model.I = pyo.RangeSet(i) # set of trips
    model.T = pyo.RangeSet(t) # set of timesteps
    model.K = pyo.RangeSet(k) # set of buses
    model.N = pyo.RangeSet(n) # set of chargers

    # parameters
    model.PI = pyo.Param(model.T, initialize=lambda model, t: PI[t-1])  # spot market electricity prices
    model.Q_begin = pyo.Param(model.P, initialize=lambda model, p: Q_begin[p-1]) # Period Q_p starting time
    model.Q_end = pyo.Param(model.P, initialize=lambda model, p: Q_end[p-1]) # Period Q_p ending time
    model.Q_len = pyo.Param(model.P, initialize=lambda model, p: Q_len[p-1]) # Period Q_p length
    model.X_low = pyo.Param(model.P, initialize=lambda model, p: X_low[p-1]) # Mininum value price x can achieve
    model.X_up = pyo.Param(model.P, initialize=lambda model, p: X_up[p-1]) # Maxinum value price x can achieve
    model.X_avg = pyo.Param(initialize=(X_avg)) # Average price x can achieve

    model.T_start = pyo.Param(model.I, initialize=lambda model, i: T_start[i-1]) # start time of trip i
    model.T_end = pyo.Param(model.I, initialize=lambda model, i: T_end[i-1]) # end time of trip i
    model.alpha = pyo.Param(model.N, initialize=lambda model, n: alpha[n-1]) # charging power of charger n
    model.ch_eff = pyo.Param(initialize=ch_eff) # charging efficiency of charger n
    model.gama = pyo.Param(model.I, initialize=lambda model, i: gama[i-1],mutable=True) # energy consumption
    model.E_0 = pyo.Param(initialize=E_0) # initial energy level of bus k
    model.E_min = pyo.Param(initialize=E_min) # minimum energy level allowed for bus k
    model.E_max = pyo.Param(initialize=E_max) # maximum energy level allowed for bus k
    model.E_end = pyo.Param(initialize=E_end) # minimum energy after an operation day for bus k
    model.C_bat = pyo.Param(model.K, initialize=lambda model, k: C_bat[k-1]) # total capacity of the bus k battery


    ### ROBUST ENERGY ###
    model.gama_deviation = pyo.Param(model.I, initialize=lambda model, i: gama_deviation[i-1]) # energy consumption
    model.gama_capital = pyo.Param(initialize=gama_capital) # robust budget
    #####################


    # non-negative variables
    model.e = pyo.Var(model.K, model.T, within=pyo.NonNegativeReals) # energy level of bus k at time t
    model.w_buy = pyo.Var(model.T, within=pyo.NonNegativeReals) # electricity purchased from the grid at time t
    model.x = pyo.Var(model.P, domain=pyo.NonNegativeIntegers) # electricity price x at period p

    # binary variables
    model.b = pyo.Var(model.K,model.I, model.T, within=pyo.Binary) # binary variable indicating if bus k is serving trip i at time t
    model.y = pyo.Var(model.K, model.N, model.T, domain=pyo.Binary) # binary variable indicating if bus k is charging
    model.c = pyo.Var(model.K, model.T, domain=pyo.Binary)  # binary variable indicating if bus k is parked to charge at time t

    #dual variables
    model.p = pyo.Var(model.T, within=pyo.NonNegativeReals)
    model.q = pyo.Var(model.T, model.I, within=pyo.NonNegativeReals)

    # constraints
    model.constraints = pyo.ConstraintList()

    ### UL ###
    # constraint 2
    for p in model.P:
        model.constraints.add(model.x[p] >= model.X_low[p])

    for p in model.P:
        model.constraints.add(model.x[p] <= model.X_up[p])

    # constraint 3
    model.constraints.add(((1/T_final) * sum(model.Q_len[p] * model.x[p] for p in model.P)) <= model.X_avg)

    ### LL ###
    #constraint 5
    for k in model.K:
        for t in model.T:
            model.constraints.add(sum(model.b[k,i,t] for i in model.I) + model.c[k,t] <=1)

    #constraint 6
    for i in model.I: 
        for t in range(model.T_start[i],model.T_end[i]):
            model.constraints.add(sum(model.b[k,i,t] for k in model.K) == 1)

    #constraint 7
    for i in model.I:
        for k in model.K:
            for t in range(model.T_start[i],model.T_end[i]-1):
                model.constraints.add(model.b[k,i,t+1] >= model.b[k,i,t])

    #constraint 8
    for n in model.N:
        for t in model.T:
            model.constraints.add(sum(model.y[k,n,t] for k in model.K) <= 1)

    #constraint 9
    for k in model.K:
        for t in model.T:
            model.constraints.add(sum(model.y[k,n,t] for n in model.N) <= model.c[k,t])

    #constraint 11
    for t in model.T:
        model.constraints.add(sum(model.ch_eff*model.alpha[n]*model.y[k,n,t] for n in model.N for k in model.K) == model.w_buy[t])

    #constraint 14
    for k in model.K:
        for t in model.T:
            model.constraints.add(model.e[k,t] >= model.C_bat[k] * model.E_min)

    #constrait 15
    for k in model.K:
        for t in model.T:
            model.constraints.add(E_max * model.C_bat[k] >= model.e[k,t] + sum(model.ch_eff*model.alpha[n]*model.y[k,n,t] for n in model.N))          

    #constraint 16
    for k in model.K:
        model.constraints.add(model.e[k,1] == model.E_0*model.C_bat[k])

    #constraint 17
    for k in model.K:
        model.constraints.add(model.e[k,T_final-1] + sum(model.ch_eff*model.alpha[n]*model.y[k,n,t] for n in model.N) >= model.E_end*model.C_bat[k])

    # extras
    #for k in model.K:
    #    model.constraints.add(sum(model.ch_eff*model.alpha[n]*model.y[k,n,1] for n in model.N)==0)

    #constraint 31
    for k in model.K:
        for t in range(2,T_final+1):
            model.constraints.add(model.e[k,t-1] - model.e[k,t] + sum(model.ch_eff*model.alpha[n]*model.y[k,n,t] for n in model.N) - sum(model.gama[i]*model.b[k,i,t] for i in model.I) == ((model.p[t] * model.gama_capital) + sum(model.q[t,i] for i in model.I)))

    #constraint 32
    for i in model.I:
        for k in model.K:
            for t in range(model.T_start[i],model.T_end[i]):
                model.constraints.add(model.p[t] + model.q[t,i] >= model.gama_deviation[i])


    # objective function
    def rule_obj(mod):
        return sum(mod.x[p] * mod.w_buy[t] for p in mod.P for t in range(mod.Q_begin[p],mod.Q_end[p])) - sum(mod.PI[t] * mod.w_buy[t] for t in model.T)
    model.obj = pyo.Objective(rule=rule_obj, sense=pyo.maximize)

    try:
        opt = pyo.SolverFactory('gurobi')
        opt.options['timelimit'] = 60
        opt.options['mipgap'] = 0.05
        results = opt.solve(model,tee=True)
    except:
        try:
            SolverFactory('mindtpy').solve(model, time_limit = 60, mip_solver='gurobi', nlp_solver='ipopt', tee=True)       
        except Exception:
            pass
            print('\n##############----------- Ignored Exception -----------##############')
    return model

## HPR

In [4]:
def solveHRP(data,y_l):
    model = pyo.ConcreteModel() # create model
    
    mult = 100 # <--------------- multiplication factor for making the variable integer

    # data
    t = len(data['dataset']['Time'])
    PI = data['dataset']['Spot Market'].values.flatten() * mult

    X_low = data['average']['MIN'].values.flatten() * mult
    X_up = data['average']['MAX'].values.flatten() * mult
    X_avg = 5

    p = len(data['periods']['PERIOD'])
    Q_begin = data['periods']['BEGIN'].tolist()
    Q_end = data['periods']['END'].tolist()
    Q_len = data['periods']['LEN'].tolist() 

    i = len(data['Trip time']['Time begin (min)'])
    k = len(data['Buses']['Bus (kWh)'])
    n = len(data['Chargers']['Charger (kWh/min)'])
    T_start = data['Trip time']['Time begin (min)'].tolist()
    T_start = [int(x) for x in T_start]
    T_end = data['Trip time']['Time finish (min)'].tolist()
    T_end = [int(x) for x in T_end]
    alpha = data['Chargers']['Charger (kWh/min)'].tolist()
    gama = data['Energy consumption']['Uncertain energy (kWh/km*min)'].tolist()
    
    ch_eff = 0.90
    E_0 = 0.2
    E_min = 0.2
    E_max = 1
    E_end = 0.2
    C_bat = data['Buses']['Bus (kWh)'].tolist()
    T_final = t

    ### ROBUST ENERGY ###
    
    gama_deviation = data['Energy consumption']['Maximum deviation (kWh/km*min)'].tolist()
    gama_capital = 0 # 0 < Gama < number of routes (worst case)
    
    #####################

    # sets
    model.P = pyo.RangeSet(p) # set of periods
    model.I = pyo.RangeSet(i) # set of trips
    model.T = pyo.RangeSet(t) # set of timesteps
    model.K = pyo.RangeSet(k) # set of buses
    model.N = pyo.RangeSet(n) # set of chargers

    # parameters
    model.PI = pyo.Param(model.T, initialize=lambda model, t: PI[t-1])  # spot market electricity prices
    model.Q_begin = pyo.Param(model.P, initialize=lambda model, p: Q_begin[p-1]) # Period Q_p starting time
    model.Q_end = pyo.Param(model.P, initialize=lambda model, p: Q_end[p-1]) # Period Q_p ending time
    model.Q_len = pyo.Param(model.P, initialize=lambda model, p: Q_len[p-1]) # Period Q_p length
    model.X_low = pyo.Param(model.P, initialize=lambda model, p: X_low[p-1]) # Mininum value price x can achieve
    model.X_up = pyo.Param(model.P, initialize=lambda model, p: X_up[p-1]) # Maxinum value price x can achieve
    model.X_avg = pyo.Param(initialize=(X_avg)) # Average price x can achieve

    model.T_start = pyo.Param(model.I, initialize=lambda model, i: T_start[i-1]) # start time of trip i
    model.T_end = pyo.Param(model.I, initialize=lambda model, i: T_end[i-1]) # end time of trip i
    model.alpha = pyo.Param(model.N, initialize=lambda model, n: alpha[n-1]) # charging power of charger n
    model.ch_eff = pyo.Param(initialize=ch_eff) # charging efficiency of charger n
    model.gama = pyo.Param(model.I, initialize=lambda model, i: gama[i-1],mutable=True) # energy consumption
    model.E_0 = pyo.Param(initialize=E_0) # initial energy level of bus k
    model.E_min = pyo.Param(initialize=E_min) # minimum energy level allowed for bus k
    model.E_max = pyo.Param(initialize=E_max) # maximum energy level allowed for bus k
    model.E_end = pyo.Param(initialize=E_end) # minimum energy after an operation day for bus k
    model.C_bat = pyo.Param(model.K, initialize=lambda model, k: C_bat[k-1]) # total capacity of the bus k battery

    model.y_l = pyo.Param(model.T, initialize=y_l)


    ### ROBUST ENERGY ###
    model.gama_deviation = pyo.Param(model.I, initialize=lambda model, i: gama_deviation[i-1]) # energy consumption
    model.gama_capital = pyo.Param(initialize=gama_capital) # robust budget
    #####################


    # non-negative variables
    model.e = pyo.Var(model.K, model.T, within=pyo.NonNegativeReals) # energy level of bus k at time t
    model.w_buy = pyo.Var(model.T, within=pyo.NonNegativeReals) # electricity purchased from the grid at time t
    model.x = pyo.Var(model.P, domain=pyo.NonNegativeIntegers) # electricity price x at period p

    # binary variables
    model.b = pyo.Var(model.K,model.I, model.T, within=pyo.Binary) # binary variable indicating if bus k is serving trip i at time t
    model.y = pyo.Var(model.K, model.N, model.T, domain=pyo.Binary) # binary variable indicating if bus k is charging
    model.c = pyo.Var(model.K, model.T, domain=pyo.Binary)  # binary variable indicating if bus k is parked to charge at time t

    #dual variables
    model.p = pyo.Var(model.T, within=pyo.NonNegativeReals)
    model.q = pyo.Var(model.T, model.I, within=pyo.NonNegativeReals)

    # constraints
    model.constraints = pyo.ConstraintList()
    
    ### UL ###
    # constraint 2
    for p in model.P:
        model.constraints.add(model.x[p] >= model.X_low[p])
    
    for p in model.P:
        model.constraints.add(model.x[p] <= model.X_up[p])

    # constraint 3
    model.constraints.add(((1/T_final) * sum(model.Q_len[p] * model.x[p] for p in model.P)) <= model.X_avg)

    ### LL ###
    #constraint 5
    for k in model.K:
        for t in model.T:
            model.constraints.add(sum(model.b[k,i,t] for i in model.I) + model.c[k,t] <=1)

    #constraint 6
    for i in model.I: 
        for t in range(model.T_start[i],model.T_end[i]):
            model.constraints.add(sum(model.b[k,i,t] for k in model.K) == 1)

    #constraint 7
    for i in model.I:
        for k in model.K:
            for t in range(model.T_start[i],model.T_end[i]-1):
                model.constraints.add(model.b[k,i,t+1] >= model.b[k,i,t])

    #constraint 8
    for n in model.N:
        for t in model.T:
            model.constraints.add(sum(model.y[k,n,t] for k in model.K) <= 1)

    #constraint 9
    for k in model.K:
        for t in model.T:
            model.constraints.add(sum(model.y[k,n,t] for n in model.N) <= model.c[k,t])

    #constraint 11
    for t in model.T:
        model.constraints.add(sum(model.ch_eff*model.alpha[n]*model.y[k,n,t] for n in model.N for k in model.K) == model.w_buy[t])

    #constraint 14
    for k in model.K:
        for t in model.T:
            model.constraints.add(model.e[k,t] >= model.C_bat[k] * model.E_min)

    #constrait 15
    for k in model.K:
        for t in model.T:
            model.constraints.add(E_max * model.C_bat[k] >= model.e[k,t] + sum(model.ch_eff*model.alpha[n]*model.y[k,n,t] for n in model.N))          

    #constraint 16
    for k in model.K:
        model.constraints.add(model.e[k,1] == model.E_0*model.C_bat[k])

    #constraint 17
    for k in model.K:
        model.constraints.add(model.e[k,T_final-1] + sum(model.ch_eff*model.alpha[n]*model.y[k,n,t] for n in model.N) >= model.E_end*model.C_bat[k])

   #constraint 18
    model.constraints.add(sum(model.x[p] * model.w_buy[t] for p in model.P for t in range(model.Q_begin[p],model.Q_end[p])) <= sum(model.x[p] * model.y_l[t] for p in model.P for t in range(model.Q_begin[p],model.Q_end[p])))

    # extras
    #for k in model.K:
    #    model.constraints.add(sum(model.ch_eff*model.alpha[n]*model.y[k,n,1] for n in model.N)==0)
    
    #constraint 31
    for k in model.K:
        for t in range(2,T_final+1):
            model.constraints.add(model.e[k,t-1] - model.e[k,t] + sum(model.ch_eff*model.alpha[n]*model.y[k,n,t] for n in model.N) - sum(model.gama[i]*model.b[k,i,t] for i in model.I) == ((model.p[t] * model.gama_capital) + sum(model.q[t,i] for i in model.I)))

    #constraint 32
    for i in model.I:
        for k in model.K:
            for t in range(model.T_start[i],model.T_end[i]):
                model.constraints.add(model.p[t] + model.q[t,i] >= model.gama_deviation[i])
 

    # objective function
    def rule_obj(mod):
        return sum(mod.x[p] * mod.w_buy[t] for p in mod.P for t in range(mod.Q_begin[p],mod.Q_end[p])) - sum(mod.PI[t] * mod.w_buy[t] for t in model.T)
    model.obj = pyo.Objective(rule=rule_obj, sense=pyo.maximize)

    try:
        opt = pyo.SolverFactory('gurobi')
        opt.options['timelimit'] = 3600
        opt.options['mipgap'] = 0.01
        results = opt.solve(model,tee=True)
    except:
        try:
            SolverFactory('mindtpy').solve(model, mip_solver='gurobi', nlp_solver='ipopt', tee=True) 
        except Exception:
            pass
            print('\n##############----------- Ignored Exception -----------##############')
    return model

## LL

In [5]:

def solveLL(data,x):
    
    model = pyo.ConcreteModel() # create model

    # data
    i = len(data['Trip time']['Time begin (min)'])
    k = len(data['Buses']['Bus (kWh)'])
    n = len(data['Chargers']['Charger (kWh/min)'])
    t = 96
    p = 7

    T_start = data['Trip time']['Time begin (min)'].tolist()
    T_start = [int(x) for x in T_start]
    T_end = data['Trip time']['Time finish (min)'].tolist()
    T_end = [int(x) for x in T_end]
    Q_end = data['periods']['END'].tolist()
    Q_begin = data['periods']['BEGIN'].tolist()
    
    ######
    PI = data['dataset']['Spot Market'].values.flatten() * 100
    ######

    alpha = data['Chargers']['Charger (kWh/min)'].tolist()
    gama = data['Energy consumption']['Uncertain energy (kWh/km*min)'].tolist()
    ch_eff = 0.90

    E_0 = 0.2
    E_min = 0.2
    E_max = 1
    E_end = 0.2
    C_bat = data['Buses']['Bus (kWh)'].tolist()

    T_final = t
    
    ### ROBUST ENERGY ###
    
    gama_deviation = data['Energy consumption']['Maximum deviation (kWh/km*min)'].tolist()
    gama_capital = 0 # 0 < Gama < number of routes (worst case)
    
    #####################

    # sets
    model.I = pyo.RangeSet(i) # set of trips
    model.T = pyo.RangeSet(t) # set of timesteps
    model.K = pyo.RangeSet(k) # set of buses
    model.N = pyo.RangeSet(n) # set of chargers
    model.P = pyo.RangeSet(p) # number of price periods

    # parameters
    model.T_start = pyo.Param(model.I, initialize=lambda model, i: T_start[i-1]) # start time of trip i
    model.T_end = pyo.Param(model.I, initialize=lambda model, i: T_end[i-1]) # end time of trip i
    model.Q_begin = pyo.Param(model.P, initialize=lambda model, p: Q_begin[p-1])
    model.Q_end = pyo.Param(model.P, initialize=lambda model, p: Q_end[p-1])
    model.alpha = pyo.Param(model.N, initialize=lambda model, n: alpha[n-1]) # charging power of charger n
    model.ch_eff = pyo.Param(initialize=ch_eff) # charging efficiency of charger n
    
    ######
    model.PI = pyo.Param(model.T, initialize=lambda model, t: PI[t-1])  # spot market electricity prices
    ######

    ### ROBUST ENERGY ###
    model.gama_deviation = pyo.Param(model.I, initialize=lambda model, i: gama_deviation[i-1]) # energy consumption
    model.gama_capital = pyo.Param(initialize=gama_capital) # robust budget
    #####################


    #model.x = pyo.Param(model.P, initialize=lambda model, p: x[p-1]) # electricity purchasing price in time t
    model.x = pyo.Param(model.P, initialize=x) # electricity purchasing price in time t

    model.gama = pyo.Param(model.I, initialize=lambda model, i: gama[i-1]) # energy consumption
    model.E_0 = pyo.Param(initialize=E_0) # initial energy level of bus k
    model.E_min = pyo.Param(initialize=E_min) # minimum energy level allowed for bus k
    model.E_max = pyo.Param(initialize=E_max) # maximum energy level allowed for bus k
    model.E_end = pyo.Param(initialize=E_end) # minimum energy after an operation day for bus k
    model.C_bat = pyo.Param(model.K, initialize=lambda model, k: C_bat[k-1]) # total capacity of the bus k battery
    

    # binary variables
    model.b = pyo.Var(model.K,model.I, model.T, within=pyo.Binary) # binary variable indicating if bus k is serving trip i at time t
    model.y = pyo.Var(model.K, model.N, model.T, domain=pyo.Binary) # binary variable indicating if bus k is occupying a charger n at time t to discharge
    model.c = pyo.Var(model.K, model.T, domain=pyo.Binary)  # binary variable indicating if bus k is charging/discharging at time t

    # non-negative variables
    model.e = pyo.Var(model.K, model.T, within=pyo.NonNegativeReals) # energy level of bus k at time t
    model.w_buy = pyo.Var(model.T, within=pyo.NonNegativeReals) # electricity purchased from the grid at time t

    #dual variables
    model.p = pyo.Var(model.T, within=pyo.NonNegativeReals)
    model.q = pyo.Var(model.T, model.I, within=pyo.NonNegativeReals)

    # constraints
    model.constraints = pyo.ConstraintList()
    
    #constraint 5
    for k in model.K:
        for t in model.T:
            model.constraints.add(sum(model.b[k,i,t] for i in model.I) + model.c[k,t] <=1)

    #constraint 6
    for i in model.I: 
        for t in range(model.T_start[i],model.T_end[i]):
            model.constraints.add(sum(model.b[k,i,t] for k in model.K) == 1)

    #constraint 7
    for i in model.I:
        for k in model.K:
            for t in range(model.T_start[i],model.T_end[i]-1):
                model.constraints.add(model.b[k,i,t+1] >= model.b[k,i,t])

    #constraint 8
    for n in model.N:
        for t in model.T:
            model.constraints.add(sum(model.y[k,n,t] for k in model.K) <= 1)

    #constraint 9
    for k in model.K:
        for t in model.T:
            model.constraints.add(sum(model.y[k,n,t] for n in model.N) <= model.c[k,t])

    #constraint 10
    for k in model.K:
        for t in range(2,T_final+1):
            model.constraints.add(model.e[k,t] == model.e[k,t-1] + sum(model.ch_eff*model.alpha[n]*model.y[k,n,t] for n in model.N) - sum(model.gama[i]*model.b[k,i,t] for i in model.I))
    
    #constraint 11
    for t in model.T:
        model.constraints.add(sum(model.ch_eff*model.alpha[n]*model.y[k,n,t] for n in model.N for k in model.K) == model.w_buy[t])

    #constraint 14
    for k in model.K:
        for t in model.T:
            model.constraints.add(model.e[k,t] >= model.C_bat[k] * model.E_min)

    #constrait 15
    for k in model.K:
        for t in model.T:
            model.constraints.add(E_max * model.C_bat[k] >= model.e[k,t] + sum(model.ch_eff*model.alpha[n]*model.y[k,n,t] for n in model.N))          

    #constraint 16
    for k in model.K:
        model.constraints.add(model.e[k,1] == model.E_0*model.C_bat[k])

    #constraint 17
    for k in model.K:
        model.constraints.add(model.e[k,T_final-1] + sum(model.ch_eff*model.alpha[n]*model.y[k,n,t] for n in model.N) >= model.E_end*model.C_bat[k]) 

    #constraint 31
    for k in model.K:
        for t in range(2,T_final+1):
            model.constraints.add(model.e[k,t-1] - model.e[k,t] + sum(model.ch_eff*model.alpha[n]*model.y[k,n,t] for n in model.N) - sum(model.gama[i]*model.b[k,i,t] for i in model.I) == ((model.p[t] * model.gama_capital) + sum(model.q[t,i] for i in model.I)))

    #constraint 32
    for i in model.I:
        for k in model.K:
            for t in range(model.T_start[i],model.T_end[i]):
                model.constraints.add(model.p[t] + model.q[t,i] >= model.gama_deviation[i])

    # objective function
    def rule_obj(mod):
        return sum(mod.x[p] * mod.w_buy[t] for p in mod.P for t in range(model.Q_begin[p],model.Q_end[p]))
    model.obj = pyo.Objective(rule=rule_obj, sense=pyo.minimize)

    try:
        opt = pyo.SolverFactory('gurobi')
        opt.options['timelimit'] = 3600
        opt.options['mipgap'] = 0.01
        results = opt.solve(model,tee=True)
    except Exception:
        pass
        print('\n##############----------- Ignored Exception -----------##############') 
    return model

## SOLVING

In [6]:
LB = float('-inf')
model_INIT = INIT(data)
UB = pyo.value(model_INIT.obj())
k = 0
x_k = model_INIT.x
y_u = model_INIT.w_buy
epsilon = 0.0001

KeyError: 'dataset'

In [ ]:
while(UB != LB):
    print('Starting iteration: ',k)
    
    if k != 0:
        pass
        model_HRP = solveHRP(data,y_l)
        x_k = model_HRP.x
        y_u = model_HRP.w_buy
        UB = model_HRP.obj()
    
    model_LL = solveLL(data,x_k)
    y_l = model_LL.w_buy
    f_y_u = sum(x_k[p] * y_u[t] for p in model_LL.P for t in range(model_LL.Q_begin[p],model_LL.Q_end[p]))

    if pyo.value(f_y_u) - model_LL.obj() <= epsilon:
        LB = UB
    else:
        F_y_l = sum(x_k[p] * y_l[t] for p in model_LL.P for t in range(model_LL.Q_begin[p],model_LL.Q_end[p])) - sum(model_LL.PI[t] * y_l[t] for t in model_LL.T)
        if pyo.value(F_y_l) > LB:
            LB = pyo.value(F_y_l)
    print('Upper bound:',UB,'\nLower Bound',LB,'\n')
    k = k+1

In [ ]:
x_data = []
spot_data = []
min_data = []
max_data = []

for p in model_HRP.P:
    for t in range(model_HRP.Q_begin[p], model_HRP.Q_end[p]):    
        x_data.append(pyo.value(x_k[p]))
        spot_data.append(pyo.value(model_HRP.PI[t]))
        min_data.append(pyo.value(model_HRP.X_low[p]))
        max_data.append(pyo.value(model_HRP.X_up[p]))

df = pd.DataFrame({'x': x_data, 'spot': spot_data, 'min': min_data, 'max': max_data})

# Plot the data
plt.plot(df.index, df['spot'], label='Spot Market')
plt.plot(df.index, df['min'], label='Mininum')
plt.plot(df.index, df['max'], label='Maximum')
plt.plot(df.index, df['x'], label='Energy Price', linestyle='--')
plt.xlabel('Time step [15min]')
plt.ylabel('Energy price [$/Kwh]')
plt.title('Energy Selling Price')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
def energy_bus(K,T,e):
    bus_list = []
    energy_list = []
    for k in K:
        bus_number = 'bus' + ' ' + str(k)
        bus_list.append(bus_number)
    for t in T:
        for  k in K:
            energy_list.append(pyo.value(e[k,t]))
    energy_array = np.reshape(energy_list, (len(T), len(bus_list)))
    Energy = pd.DataFrame(energy_array,index=T, columns=bus_list)
    return Energy

def power(T,w):
    transac_list = []
    for t in T:
        value = pyo.value(w[t])
        transac_list.append(value)
    W = pd.DataFrame(transac_list, index=model_LL.T, columns=['Power'])
    return W

C_bat = data['Buses']['Bus (kWh)'].tolist()
Energy = energy_bus(model_HRP.K,model_HRP.T,model_HRP.e)
Energy_perc = (Energy*100)/C_bat[0]
Energy_perc.plot(figsize=(12,6))
plt.xlabel('Time [min]')
plt.ylabel('State of Charge [%]')

W_buy = power(model_HRP.T,y_u)
W_buy.plot(figsize=(12,6))
plt.xlabel('Time [min]')
plt.ylabel('Power [kWh]')

## VISUALIZATION

In [ ]:
# Values PTO
buy = sum(model_LL.pho_plus[p]*model_LL.w_buy[t] for p in model_LL.P for t in range(model_LL.Q_begin[p], model_LL.Q_end[p]))
sell = sum(model_LL.pho_minus[p]*model_LL.w_sell[t] for p in model_LL.P for t in range(model_LL.Q_begin[p], model_LL.Q_end[p]))
degra = sum(model_LL.d[k, t] for k in model_LL.K for t in model_LL.T)
power = sum(model_LL.U_price[l]*model_LL.u[l] for l in model_LL.L)
cap = sum(model_LL.mi[p]*model_LL.w_cap[t] for p in model_LL.P for t in range(model_LL.Q_begin[p], model_LL.Q_end[p]))
costs = buy - sell + degra + power - cap
print('Values from the PTO side:\n'
    'Total costs:',pyo.value(costs),'\n'
    'Energy bought from the Aggregator:',pyo.value(buy),'\n'
    'Energy sold to the Aggregator:',pyo.value(sell),'\n'
    'Battery degradation costs:',pyo.value(degra),'\n'
    'Power bought:',pyo.value(power),'\n'
    'FCR offered to the grid:',pyo.value(cap)
)

In [ ]:
#Values Aggregator
sell_to_PTO = sum(pho_plus[p] * y_buy_l[t] for p in model_HRP.P for t in range(model_HRP.Q_begin[p], model_HRP.Q_end[p]))
buy_from_PTO = sum(pho_minus[p] * y_sell_l[t] for p in model_HRP.P for t in range(model_HRP.Q_begin[p], model_HRP.Q_end[p]))
cap_from_PTO = sum(mi[p] * y_cap_l[t] for p in model_HRP.P for t in range(model_HRP.Q_begin[p], model_HRP.Q_end[p]))
cap_to_GRID = sum(model_HRP.PI_cap[t] * y_cap_l[t] for p in model_HRP.P for t in range(model_HRP.Q_begin[p], model_HRP.Q_end[p]))
buy_from_GRID = sum(model_HRP.PI[t] * y_buy_l[t] for t in model_HRP.T)
sell_to_GRID = sum(model_HRP.PI[t] * y_sell_l[t] for t in model_HRP.T)
revenues = sell_to_PTO - buy_from_PTO - cap_from_PTO + cap_to_GRID - buy_from_GRID + sell_to_GRID

print('Values from the Aggregator side:\n'
    'Total revenues:',pyo.value(revenues),'\n'
    'Energy sold to the PTO:',pyo.value(sell_to_PTO),'\n'
    'Energy bought from the PTO:',pyo.value(buy_from_PTO),'\n'
    'FCR bought from the PTO:',pyo.value(cap_from_PTO),'\n'
    'FCR offered to the grid:',pyo.value(cap_to_GRID),'\n'
    'Energy bought in the Wholesale market:',pyo.value(buy_from_GRID),'\n'
    'Energy sold to the grid:',pyo.value(sell_to_GRID))

In [ ]:
# Set Seaborn style
sns.set(style="ticks")

model = model_LL

# Initialize empty lists
pho_plus_data = []
pho_minus_data = []
mi_data = []
spot_data = []
min_data = []
max_data = []
mi_min_data = []
mi_max_data = []

# Getting prices information
for p in model_HRP.P:
    for t in range(model_HRP.Q_begin[p], model_HRP.Q_end[p]):
        pho_plus_data.append(pyo.value(pho_plus[p]))
        pho_minus_data.append(pyo.value(pho_minus[p]))
        mi_data.append(pyo.value(mi[p]))
        spot_data.append(pyo.value(model_HRP.PI[t]))
        min_data.append(pyo.value(model_HRP.X_low[p]))
        max_data.append(pyo.value(model_HRP.X_up[p]))  
        mi_min_data.append(pyo.value(model_HRP.Mi_low[p]))
        mi_max_data.append(pyo.value(model_HRP.Mi_up[p]))

df = pd.DataFrame({'sell': pho_plus_data, 'buy': pho_minus_data,
                  'cap': mi_data, 'spot': spot_data, 'min': min_data, 'max': max_data, 'min_cap': mi_min_data, 'max_cap': mi_max_data})

def extract_model_data(model):
    w_buy = [model.w_buy[t]() * 4 for t in model.T]
    w_sell = [model.w_sell[t]() * 4 for t in model.T]
    w_cap = [model.w_cap[t]() for t in model.T]
    e_values = np.array([[model.e[k, t].value for t in model.T] for k in model.K])
    u_values = [model.u[l].value for l in model.L]
    x_values = np.array([[[model.x[k, n, t].value for t in model.T] for n in model.N] for k in model.K])
    y_values = np.array([[[model.y[k, n, t].value for t in model.T] for n in model.N] for k in model.K])
    z_values = np.array([[[model.z[k, n, t].value for t in model.T] for n in model.N] for k in model.K])
    c_values = np.array([[model.c[k, t].value for t in model.T] for k in model.K])
    d_values = np.array([[model.d[k, t].value for t in model.T] for k in model.K])
    b_values = np.array([[[model.b[k, i, t].value for t in model.T] for i in model.I] for k in model.K])

    return w_buy, w_sell, w_cap, e_values, u_values, x_values, y_values, z_values, c_values, d_values, b_values

# Extract model data
w_buy, w_sell, w_cap, e_values, u_values, x_values, y_values, z_values, c_values, d_values, b_values = extract_model_data(model)

# Plotting energy selling price
plt.figure(figsize=(12, 6))
sns.lineplot(data=df[['spot', 'min', 'max', 'sell', 'buy']], palette="tab10", linestyle='--', linewidth=1)
plt.xlabel('Time step [15min]', fontsize=14)
plt.ylabel('Energy price [$/kWh]', fontsize=14)
plt.title('Energy Prices', fontsize=16)
plt.grid(True, linestyle='-', alpha=0.7)
plt.legend()
plt.show()

# Plotting energy capacity price
plt.figure(figsize=(12, 6))
sns.lineplot(data=df[['min_cap', 'max_cap', 'cap']], palette="tab10", linestyle=':', linewidth=1)
plt.xlabel('Time step [15min]', fontsize=14)
plt.ylabel('Capacity price [$/kW]', fontsize=14)
plt.title('Energy Capacity Prices', fontsize=16)
plt.grid(True, linestyle='-', alpha=0.7)
plt.legend()
plt.show()

# Plotting energy levels
plt.figure(figsize=(12, 6))
sns.lineplot(data=e_values.T, palette="tab10", linewidth=2, dashes=False)
plt.xlabel('Time step', fontsize=14)
plt.ylabel('Energy [kWh]', fontsize=14)
plt.title('Energy Levels of Buses', fontsize=16)
plt.legend(title='Bus', loc='upper left')
plt.grid(True, linestyle='-', alpha=0.7)
plt.show()

# Plotting charging power
plt.figure(figsize=(12, 6))
sns.lineplot(x=range(1, len(w_buy) + 1), y=w_buy, label='Charging')
sns.lineplot(x=range(1, len(w_sell) + 1), y=w_sell, label='Discharging')
plt.xlabel('Time Step', fontsize=14)
plt.ylabel('Power [kW]', fontsize=14)
plt.title('Charging Power', fontsize=16)
plt.legend()
plt.grid(True)
plt.show()

# Plotting capacity
plt.figure(figsize=(12, 6))
sns.lineplot(x=range(1, len(w_cap) + 1), y=w_cap, color='r')
plt.xlabel('Time Step', fontsize=14)
plt.ylabel('Capacity [kW]', fontsize=14)
plt.title('Capacity', fontsize=16)
plt.grid(True)
plt.show()

In [ ]:
# Create DataFrame for the first set of data
data1 = {'sell': pho_plus_data, 'buy': pho_minus_data,
         'cap': mi_data, 'spot': spot_data, 'min': min_data, 'max': max_data, 'min_cap': mi_min_data, 'max_cap': mi_max_data}
df1 = pd.DataFrame(data1)

# Save the first DataFrame to an Excel file
with pd.ExcelWriter('output_data.xlsx') as writer:
    df1.to_excel(writer, sheet_name='Tariffs', index=False)

# Create DataFrames for the second set of data
data2 = {'w_buy': w_buy, 'w_sell': w_sell, 'w_cap': w_cap}
df2 = pd.DataFrame(data2)

# Create DataFrame for e_values
e_values_t = e_values.T  # transposing the data
e_df = pd.DataFrame(e_values_t, columns=model.K, index=model.T)

# Save the second set of DataFrames to the same Excel file, each in a separate sheet
with pd.ExcelWriter('output_data.xlsx', engine='openpyxl', mode='a') as writer:
    df2.to_excel(writer, sheet_name='Power', index=False)
    e_df.to_excel(writer, sheet_name='Bus Energy', index=True)

# Create DataFrame for the thrid set of data
dataPTO = {
    'Costs_PTO':[pyo.value(costs)],
    'sell_PTO': [pyo.value(sell)],
    'buy_PTO': [pyo.value(buy)],
    'degra_PTO': [pyo.value(degra)],
    'power_PTO': [pyo.value(power)],
    'cap_PTO': [pyo.value(cap)],
}

dataAggre = {
    'revenues_Agg': [pyo.value(revenues)],
    'sell_to_PTO_Agg': [pyo.value(sell_to_PTO)],
    'buy_from_PTO_Agg': [pyo.value(buy_from_PTO)],
    'cap_from_PTO_Agg': [pyo.value(cap_from_PTO)],
    'cap_to_GRID_Agg': [pyo.value(cap_to_GRID)],
    'buy_from_GRID_Agg': [pyo.value(buy_from_GRID)],
    'sell_to_GRID_Agg': [pyo.value(sell_to_GRID)]
}

dfPTO = pd.DataFrame(dataPTO)
dfAggre= pd.DataFrame(dataAggre)

# Save the second set of DataFrames to the same Excel file, each in a separate sheet
with pd.ExcelWriter('output_data.xlsx', engine='openpyxl', mode='a') as writer:
    dfPTO.to_excel(writer, sheet_name='PTO', index=False)
    dfAggre.to_excel(writer, sheet_name='Aggregator', index=False)